<a href="https://colab.research.google.com/github/io-uty/skt-bigdata-analysis/blob/main/00_skt_kobert_base_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
from transformers import AutoModel

# parameter check
model = AutoModel.from_pretrained("skt/kobert-base-v1")

state_dict = model.state_dict()

total_params = 0
total_bytes = 0

print(f"{'Name':80s} {'Shape':40s} {'Dtype':20s} {'Bytes'}")
print("-" * 140)

for name, param in state_dict.items():
    numel = param.numel()
    dtype = param.dtype
    bytes_per_elem = torch.finfo(dtype).bits // 8 if dtype.is_floating_point else torch.iinfo(dtype).bits // 8
    size_bytes = numel * bytes_per_elem

    total_params += numel
    total_bytes += size_bytes

    print(f"{name:80s} {str(tuple(param.shape)):40s} {str(dtype):20s} {size_bytes}")

print("-" * 140)
print(f"Total Parameters: {total_params:,}")
print(f"Estimated Memory Size: {total_bytes / (1024**2):.2f} MB")


""" =========================================================== """


# half parameters
model = model.half()

# 실제 모델의 파라미터를 담고있는 state_dict 값 분석
state_dict = model.state_dict()


total_params = 0
total_bytes = 0

print(f"{'Name':80s} {'Shape':40s} {'Dtype':20s} {'Bytes'}")
print("-" * 140)

for name, param in state_dict.items():
    numel = param.numel()
    dtype = param.dtype
    bytes_per_elem = torch.finfo(dtype).bits // 8 if dtype.is_floating_point else torch.iinfo(dtype).bits // 8
    size_bytes = numel * bytes_per_elem

    total_params += numel
    total_bytes += size_bytes

    print(f"{name:80s} {str(tuple(param.shape)):40s} {str(dtype):20s} {size_bytes}")

print("-" * 140)
print(f"Total Parameters: {total_params:,}")
print(f"Estimated Memory Size: {total_bytes / (1024 ** 2):.2f} MB")

torch.save(model.state_dict(), "fp16_kobert.pt")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Name                                                                             Shape                                    Dtype                Bytes
--------------------------------------------------------------------------------------------------------------------------------------------
embeddings.word_embeddings.weight                                                (8002, 768)                              torch.float32        24582144
embeddings.position_embeddings.weight                                            (512, 768)                               torch.float32        1572864
embeddings.token_type_embeddings.weight                                          (2, 768)                                 torch.float32        6144
embeddings.LayerNorm.weight                                                      (768,)                                   torch.float32        3072
embeddings.LayerNorm.bias                                                        (768,)                        

In [3]:
#batch 적용 안된 것
#모델 파라미터만 Total Parameters / 2^20 * 4(byte)
# * batch numbers = 기가바이트
# 만약 16이라면 *4가 아닌 *2만 해도 됨